# Variance-Aware Demosaicing Validation

This notebook tests how well the variance-aware Malvar demosaicing recovers the ground truth photoelectron image.

## Test Strategy

1. **Generate ground truth**: Simulate image with `return_normal_image=True` to get the ideal photoelectron image (flat QE, no Bayer pattern)
2. **Generate Bayer image**: Same simulation generates the Bayer-patterned ADU image  
3. **Demosaic**: Apply `variance_aware_malvar_demosaic` to recover photoelectrons from Bayer ADU
4. **Compare**: Measure how close the demosaiced result is to ground truth

**Key question**: How much signal degradation occurs due to Bayer sampling and demosaicing?

In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr
import pandas as pd

import sys

sys.path.append("../..")
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20260121_090651.log
/tmp/ipykernel_3391047/1927427140.py:32: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter()


## Setup Simulation Parameters

In [3]:
# Camera parameters
image_size_px = 256
pixel_size = 69.0  # nm

# data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/"
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

masks = M_F.get_masks(size_x=image_size_px, size_y=image_size_px)
R, G, B, wavelength = S_F.getpixelefficiency()
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size_px, image_size_px), np.median(gain))
camera_parameters["offset"] = np.full((image_size_px, image_size_px), np.median(offset))
camera_parameters["rqe"] = np.full((image_size_px, image_size_px), np.median(rqe))
camera_parameters["variance"] = np.full((image_size_px, image_size_px), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size_px, image_size_px), np.median(readnoise))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

print(f"Image size: {image_size_px}")
print(f"Pixel size: {pixel_size} nm")
print(f"Gain: {gain[0, 0]} ADU/pe")
print(f"Offset: {offset[0, 0]} ADU")
print(f"Variance: {variance[0, 0]} ADU²")

Image size: 256
Pixel size: 69.0 nm
Gain: 0.4073526859283447 ADU/pe
Offset: 32.49509811401367 ADU
Variance: 1.0198760032653809 ADU²


## Setup Dye and Spectral Properties

In [ ]:
# Get ATTO 565 spectrum
spectral = SF.Spectral_Funcs()

try:
    atto565_spectrum = spectral.get_dye_or_filter_data(['ATTO 565'], wavelength=wavelengths_nm)
    atto565_spectrum = np.squeeze(atto565_spectrum)
    print("✓ Loaded ATTO 565 spectrum from database")
except:
    print("⚠ Using fallback ATTO 565 spectrum")
    atto565_spectrum = np.exp(-((wavelengths_nm - 590)**2) / (2 * 30**2))

# Normalize
atto565_spectrum = atto565_spectrum / np.trapz(atto565_spectrum, wavelengths_nm)

# Calculate average emission wavelength
average_emission_wavelength = np.trapz(
    y=wavelengths_nm * atto565_spectrum,
    x=wavelengths_nm
)

# Calculate dye-pixel efficiency for each channel
dye_pixel_efficiency = np.zeros(3)
for i in range(3):
    dye_pixel_efficiency[i] = np.trapz(
        y=atto565_spectrum * pixel_QYs[i, :],
        x=wavelengths_nm
    )

print(f"Average emission: {average_emission_wavelength:.1f} nm")
print(f"Channel efficiencies (B/G/R): {dye_pixel_efficiency}")

## Create Molecule Grid for Testing

In [ ]:
# Create a few test molecules (simple grid)
n_spots = 3  # Small test case
pixel_size_nm = 69.0  # nm/pixel

# Simple positions: 3 molecules spaced across the image
x_coords = np.array([64*pixel_size_nm, 128*pixel_size_nm, 192*pixel_size_nm])  # nm
y_coords = np.array([128*pixel_size_nm, 64*pixel_size_nm, 192*pixel_size_nm])  # nm

# Format for simulation: (n_molecules, 2, n_frames)
# All molecules appear in a single frame
x0y0 = {'ATTO 565': np.zeros((n_spots, 2, 1))}
for i in range(n_spots):
    x0y0['ATTO 565'][i, 0, 0] = x_coords[i]  # x in nm
    x0y0['ATTO 565'][i, 1, 0] = y_coords[i]  # y in nm

# Photon counts per molecule
n_photons_per_spot = 2000
n_photons = {'ATTO 565': np.full(n_spots, n_photons_per_spot)}

print(f"Created {n_spots} molecules for quick test")
print(f"Photons per spot: {n_photons_per_spot}")
print(f"Positions (nm): {list(zip(x_coords, y_coords))}")

## Setup PSF Smoothing

In [ ]:
# Create smoothing function object
scmos = sCMOSFunctions.sCMOS_Functions()

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = scmos.gaussian_filter_stack
smoothing_function.data_arg = "image"

print("✓ Smoothing function configured")

## Generate Simulation with Ground Truth

In [ ]:
# Create simulator
simulator = MSF.MultiC_Sim_Funcs()

# Generate images with ground truth
print("Generating simulation with return_normal_image=True...")

bayer_image, smoothed_image, normal_image = simulator.gen_camera_image_stack(
    camera_calibration=camera_parameters,
    wavelength=wavelengths_nm,
    average_emission_wavelengths=average_emission_wavelength,
    dye_pixel_efficiency=dye_pixel_efficiency,
    n_photons=n_photons,
    x0y0=x0y0,
    smoothing_function=smoothing_function,
    background_photons=10.0,
    NA=1.49,
    pixel_size=pixel_size,
    return_normal_image=True
)

print(f"✓ Bayer image shape: {bayer_image.shape}")
print(f"✓ Normal image shape: {normal_image.shape}")
print(f"  Bayer range: [{bayer_image.min()}, {bayer_image.max()}] ADU")
print(f"  Normal range: [{normal_image.min()}, {normal_image.max()}] ADU")

## Convert Ground Truth to Photoelectrons

In [ ]:
# Convert normal image from ADU to photoelectrons
# Squeeze to remove frame dimension if single frame
normal_image_2d = np.squeeze(normal_image)
bayer_image_2d = np.squeeze(bayer_image)

ground_truth_pe = (normal_image_2d.astype(np.float32) - offset) / gain

print(f"Ground truth (photoelectrons):")
print(f"  Shape: {ground_truth_pe.shape}")
print(f"  Range: [{ground_truth_pe.min():.1f}, {ground_truth_pe.max():.1f}] pe")
print(f"  Mean: {ground_truth_pe.mean():.1f} pe")
print(f"  Std: {ground_truth_pe.std():.1f} pe")

## Apply Variance-Aware Demosaicing

In [ ]:
# Apply variance-aware Malvar demosaicing
print("Applying variance_aware_malvar_demosaic...")

demosaiced_rgb = scmos.variance_aware_malvar_demosaic(
    CFA=bayer_image_2d,
    variance_map=variance,
    offset_map=offset,
    gain=gain,
    grayscale=False
)

# Also get grayscale version
demosaiced_gray = scmos.variance_aware_malvar_demosaic(
    CFA=bayer_image_2d,
    variance_map=variance,
    offset_map=offset,
    gain=gain,
    grayscale=True
)

print(f"✓ Demosaiced RGB shape: {demosaiced_rgb.shape}")
print(f"✓ Demosaiced grayscale shape: {demosaiced_gray.shape}")
print(f"  RGB range: [{demosaiced_rgb.min():.1f}, {demosaiced_rgb.max():.1f}] pe")
print(f"  Grayscale range: [{demosaiced_gray.min():.1f}, {demosaiced_gray.max():.1f}] pe")

## Compare Demosaiced to Ground Truth

In [ ]:
# Calculate difference
difference = demosaiced_gray - ground_truth_pe
abs_difference = np.abs(difference)
rel_difference = abs_difference / (ground_truth_pe + 1e-6)

print("="*70)
print("COMPARISON: Demosaiced vs Ground Truth")
print("="*70)

print(f"\nAbsolute Difference (photoelectrons):")
print(f"  Mean: {abs_difference.mean():.3f} pe")
print(f"  Median: {np.median(abs_difference):.3f} pe")
print(f"  Std: {abs_difference.std():.3f} pe")
print(f"  Max: {abs_difference.max():.3f} pe")
print(f"  95th percentile: {np.percentile(abs_difference, 95):.3f} pe")

print(f"\nRelative Difference (%):")
print(f"  Mean: {rel_difference.mean()*100:.2f}%")
print(f"  Median: {np.median(rel_difference)*100:.2f}%")
print(f"  95th percentile: {np.percentile(rel_difference, 95)*100:.2f}%")

print(f"\nSignal Recovery:")
snr_gt = ground_truth_pe.mean() / ground_truth_pe.std()
snr_demosaic = demosaiced_gray.mean() / demosaiced_gray.std()
print(f"  Ground truth SNR: {snr_gt:.2f}")
print(f"  Demosaiced SNR: {snr_demosaic:.2f}")
print(f"  SNR ratio: {snr_demosaic/snr_gt:.3f}")

# Calculate RMSE
rmse = np.sqrt(np.mean(difference**2))
normalized_rmse = rmse / ground_truth_pe.mean()
print(f"\nRMSE: {rmse:.3f} pe ({normalized_rmse*100:.2f}% of mean signal)")

# Calculate correlation
correlation = np.corrcoef(ground_truth_pe.ravel(), demosaiced_gray.ravel())[0, 1]
print(f"Correlation: {correlation:.6f}")

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Bayer image (ADU)
im0 = axes[0, 0].imshow(bayer_image_2d, cmap='gray')
axes[0, 0].set_title('Input: Bayer Image (ADU)', fontsize=12, fontweight='bold')
plt.colorbar(im0, ax=axes[0, 0], label='ADU')
axes[0, 0].axis('off')

# Ground truth (photoelectrons)
im1 = axes[0, 1].imshow(ground_truth_pe, cmap='gray')
axes[0, 1].set_title('Ground Truth (photoelectrons)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[0, 1], label='pe')
axes[0, 1].axis('off')

# Demosaiced (photoelectrons)
im2 = axes[0, 2].imshow(demosaiced_gray, cmap='gray')
axes[0, 2].set_title('Demosaiced (photoelectrons)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 2], label='pe')
axes[0, 2].axis('off')

# Absolute difference
im3 = axes[1, 0].imshow(abs_difference, cmap='hot')
axes[1, 0].set_title(f'Absolute Difference (mean={abs_difference.mean():.2f} pe)', fontsize=12)
plt.colorbar(im3, ax=axes[1, 0], label='pe')
axes[1, 0].axis('off')

# Relative difference
im4 = axes[1, 1].imshow(rel_difference * 100, cmap='hot', vmax=20)
axes[1, 1].set_title(f'Relative Difference (mean={rel_difference.mean()*100:.1f}%)', fontsize=12)
plt.colorbar(im4, ax=axes[1, 1], label='%')
axes[1, 1].axis('off')

# Scatter plot: ground truth vs demosaiced
axes[1, 2].scatter(ground_truth_pe.ravel(), demosaiced_gray.ravel(), 
                   alpha=0.3, s=1, c='blue')
axes[1, 2].plot([ground_truth_pe.min(), ground_truth_pe.max()],
                [ground_truth_pe.min(), ground_truth_pe.max()],
                'r--', linewidth=2, label='Perfect recovery')
axes[1, 2].set_xlabel('Ground Truth (pe)', fontsize=11)
axes[1, 2].set_ylabel('Demosaiced (pe)', fontsize=11)
axes[1, 2].set_title(f'Correlation: {correlation:.4f}', fontsize=12)
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../unit_tests/demosaic_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to: unit_tests/demosaic_validation.png")

## Spot-by-Spot Analysis

In [ ]:
# Extract peaks from both images
from scipy.ndimage import maximum_filter

def find_peaks(image, min_distance=5, threshold_rel=0.3):
    """Find local maxima in image."""
    # Local maximum filter
    local_max = maximum_filter(image, size=min_distance) == image
    # Threshold
    threshold = threshold_rel * image.max()
    peaks = local_max & (image > threshold)
    # Get coordinates
    y, x = np.where(peaks)
    intensities = image[y, x]
    return y, x, intensities

# Find peaks
y_gt, x_gt, int_gt = find_peaks(ground_truth_pe)
y_dem, x_dem, int_dem = find_peaks(demosaiced_gray)

print(f"Ground truth: {len(y_gt)} peaks detected")
print(f"Demosaiced: {len(y_dem)} peaks detected")

# Match peaks (nearest neighbor within 3 pixels)
matched_intensities_gt = []
matched_intensities_dem = []

for i in range(len(y_gt)):
    distances = np.sqrt((x_dem - x_gt[i])**2 + (y_dem - y_gt[i])**2)
    if distances.min() < 3.0:
        j = distances.argmin()
        matched_intensities_gt.append(int_gt[i])
        matched_intensities_dem.append(int_dem[j])

matched_intensities_gt = np.array(matched_intensities_gt)
matched_intensities_dem = np.array(matched_intensities_dem)

print(f"\nMatched {len(matched_intensities_gt)} spots")
print(f"\nPeak Intensity Recovery:")
intensity_ratio = matched_intensities_dem / matched_intensities_gt
print(f"  Mean ratio: {intensity_ratio.mean():.3f} ± {intensity_ratio.std():.3f}")
print(f"  Median ratio: {np.median(intensity_ratio):.3f}")
print(f"  Range: [{intensity_ratio.min():.3f}, {intensity_ratio.max():.3f}]")

In [ ]:
# Plot peak intensity comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax1.scatter(matched_intensities_gt, matched_intensities_dem, s=50, alpha=0.6)
ax1.plot([matched_intensities_gt.min(), matched_intensities_gt.max()],
         [matched_intensities_gt.min(), matched_intensities_gt.max()],
         'r--', linewidth=2, label='Perfect recovery')
ax1.set_xlabel('Ground Truth Peak Intensity (pe)', fontsize=12)
ax1.set_ylabel('Demosaiced Peak Intensity (pe)', fontsize=12)
ax1.set_title('Peak Intensity Recovery', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram of ratios
ax2.hist(intensity_ratio, bins=20, alpha=0.7, edgecolor='black')
ax2.axvline(intensity_ratio.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {intensity_ratio.mean():.3f}')
ax2.axvline(1.0, color='green', linestyle='--', 
            linewidth=2, label='Perfect (1.0)')
ax2.set_xlabel('Intensity Ratio (Demosaiced/Ground Truth)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Distribution of Peak Recovery Ratios', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook validates the variance-aware Malvar demosaicing by comparing it to the ground truth photoelectron image (generated with flat QE, no Bayer pattern).

**Expected observations:**
- Some signal loss due to Bayer subsampling (each color channel only sees 25-50% of pixels)
- Spatial blurring from interpolation
- Channel-dependent degradation (green better than red/blue due to 2× sampling)

**Good performance indicators:**
- High correlation (> 0.95)
- Low relative error (< 10%)
- Peak intensity recovery ratio near 1.0
- Systematic (not random) errors suggesting bias that could be calibrated out